In [6]:
# Bibliotecas necessárias
import sys
import os

sys.path.append(
    os.path.abspath("../modules")
)

import pandas as pd
import joblib

In [ ]:
from feature_extractor import extract_features

In [9]:
# Carregando o modelo Random Forest treinado para URL
rf_model = joblib.load(
    "../data/output_API/models/rf_model_url.pkl"
)

In [10]:
url_features = [

    'NumDots',
    'SubdomainLevel',
    'PathLevel',
    'UrlLength',
    'NumDash',
    'NumDashInHostname',
    'AtSymbol',
    'TildeSymbol',
    'NumUnderscore',
    'NumPercent',
    'NumQueryComponents',
    'NumAmpersand',
    'NumHash',
    'NumNumericChars',
    'NoHttps',
    'IpAddress',
    'DomainInSubdomains',
    'DomainInPaths',
    'HostnameLength',
    'PathLength',
    'QueryLength',
    'DoubleSlashInPath'
]

In [11]:
def predict_url(url):

    # Extração
    features = extract_features(url)

    # DataFrame
    df = pd.DataFrame([features])

    # Ordenando colunas
    df = df[url_features]

    # Probabilidades
    probabilities = rf_model.predict_proba(df)[0]

    legit_prob = probabilities[0]
    phishing_prob = probabilities[1]

    # =========================
    # Classificação de risco
    # =========================

    if phishing_prob < 0.30:

        risk = "Baixo Risco"
        prediction = "Legítima"

    elif phishing_prob < 0.60:

        risk = "Risco Moderado"
        prediction = "Suspeita"

    else:

        risk = "Alto Risco"
        prediction = "Phishing"

    # =========================
    # Retorno
    # =========================

    return {

        "url": url,

        "prediction": prediction,

        "legitimate_probability": float(
            round(legit_prob, 4) * 100 
        ),

        "phishing_probability": float(
            round(phishing_prob, 4) * 100
        ),

        "risk_level": risk
    }

In [12]:
result = predict_url(
    "http://paypal-login-security-update.com/verify?id=12345"
)

result

{'url': 'http://paypal-login-security-update.com/verify?id=12345',
 'prediction': 'Legítima',
 'legitimate_probability': 75.0,
 'phishing_probability': 25.0,
 'risk_level': 'Baixo Risco'}

In [13]:
result = predict_url(
    "http://paypal-login-secure.com/login?id=123"
)

result

{'url': 'http://paypal-login-secure.com/login?id=123',
 'prediction': 'Legítima',
 'legitimate_probability': 77.0,
 'phishing_probability': 23.0,
 'risk_level': 'Baixo Risco'}

In [14]:
result = predict_url(
    "http://paypal-login-security-update.com/verify/account?id=12345"
)

result

{'url': 'http://paypal-login-security-update.com/verify/account?id=12345',
 'prediction': 'Suspeita',
 'legitimate_probability': 66.0,
 'phishing_probability': 34.0,
 'risk_level': 'Risco Moderado'}

In [15]:
result = predict_url(
    "http://192.168.0.1/verify/login/update?id=99999"
)

result

{'url': 'http://192.168.0.1/verify/login/update?id=99999',
 'prediction': 'Suspeita',
 'legitimate_probability': 42.0,
 'phishing_probability': 57.99999999999999,
 'risk_level': 'Risco Moderado'}

In [16]:
result = predict_url(
    "http://255.255.255.255/paypal/secure/login/update?id=9999"
)

result

{'url': 'http://255.255.255.255/paypal/secure/login/update?id=9999',
 'prediction': 'Suspeita',
 'legitimate_probability': 53.0,
 'phishing_probability': 47.0,
 'risk_level': 'Risco Moderado'}